# Notebook 02: Structure-Aware Chunking & Vector Embeddings

**Hybrid-Agentic-RAG Capstone Demonstration**  
**Domain:** Technical Enterprise Documentation (Docker & Kubernetes Infrastructure)

This notebook demonstrates Phase 2 & Phase 3 of the Capstone:
1. **Structure-Aware Chunking:** Dividing documents into 300–500 token chunks while preserving 15% overlap and rich heading metadata.
2. **Vector Embeddings:** Generating dense vector representations using `BAAI/bge-small-en-v1.5` (384 dimensions) and computing semantic cosine similarity.

## 1. Environment & Project Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np

# Resolve repository root dynamically
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Project root resolved: {repo_root}")

## 2. Structure-Aware Chunking Demonstration
Unlike standard character-count splitters that break sentences and code blocks mid-line, the project's `StructureAwareChunker` operates with structural element awareness targeting **300 to 500 tokens with 15% overlap**.

In [ ]:
from src.ingestion.loaders.factory import get_loader
from src.ingestion.chunker import StructureAwareChunker

# Load a sample document
sample_file = repo_root / "data" / "raw" / "docker-bridge-network.html"
loader = get_loader(sample_file)
raw_doc = loader.load(sample_file)

# Initialize chunker with production parameters
chunker = StructureAwareChunker(
    min_tokens=300,
    max_tokens=500,
    overlap_ratio=0.15
)

chunks = chunker.chunk_document(raw_doc)
print(f"Document '{raw_doc.filename}' split into {len(chunks)} structure-aware chunks.\n")

# Inspect token distribution
token_counts = [c.metadata.token_count for c in chunks]
print(f"Token Stats: Min={min(token_counts)}, Max={max(token_counts)}, Mean={np.mean(token_counts):.1f}")

### Inspecting Preserved Chunk Metadata
Every chunk maintains rich metadata for precise source attribution and metadata filtering.

In [ ]:
sample_chunk = chunks[min(2, len(chunks)-1)]

print(f"Chunk ID: {sample_chunk.id}")
print(f"Source File: {sample_chunk.metadata.filename}")
print(f"Format: {sample_chunk.metadata.doc_type}")
print(f"Heading: {sample_chunk.metadata.heading}")
print(f"Section Hierarchy: {sample_chunk.metadata.section}")
print(f"Tokens: {sample_chunk.metadata.token_count}")
print(f"Chars: {sample_chunk.metadata.char_count}")
print("\n--- Sample Chunk Text Content (First 250 chars) ---")
print(sample_chunk.text[:250] + "...")

## 3. Vector Embeddings (`BAAI/bge-small-en-v1.5`)
We use the embedding model specified in `src/config.py` (`BAAI/bge-small-en-v1.5`), which generates 384-dimensional dense vectors optimized for retrieval tasks.

*Note: The model is cached locally under `data/cache/huggingface` for fully offline execution.*

In [ ]:
from sentence_transformers import SentenceTransformer
from src.config import settings

print(f"Configured Embedding Model: {settings.embedding_model_name}")

# Load the model
model = SentenceTransformer(settings.embedding_model_name, cache_folder=settings.hf_home)
print(f"Embedding Model Loaded! Vector Dimensions: {model.get_sentence_embedding_dimension()}")

## 4. Semantic Similarity & Cosine Similarity Demonstration
Let's test semantic similarity between a user query and different chunks:
- **Query:** *"How do user-defined bridge networks handle container DNS resolution?"*
- **Target Chunk 1:** Chunk describing user-defined bridges and automatic DNS resolution.
- **Target Chunk 2:** An unrelated chunk or out-of-domain text.

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

query = "How do user-defined bridge networks handle container DNS resolution?"

# Find a relevant chunk mentioning DNS or user-defined bridge
relevant_chunk = next(
    (c for c in chunks if "dns" in c.text.lower() or "user-defined" in c.text.lower()),
    chunks[0]
)

# Create an unrelated control text
irrelevant_text = "Annual company financial reports indicate a 12 percent growth in quarterly recurring SaaS subscription revenue."

# Generate embeddings
q_emb = model.encode(query)
rel_emb = model.encode(relevant_chunk.text)
irrel_emb = model.encode(irrelevant_text)

score_rel = cosine_similarity(q_emb, rel_emb)
score_irrel = cosine_similarity(q_emb, irrel_emb)

print(f"User Query: '{query}'\n")
print(f"  [+] Cosine Similarity with Relevant Technical Chunk: {score_rel:.4f}")
print(f"      Heading: {relevant_chunk.metadata.heading}")
print(f"  [-] Cosine Similarity with Unrelated Financial Text:  {score_irrel:.4f}")
print(f"\nSemantic Discrimination Delta: {score_rel - score_irrel:+.4f}")

## 5. Key Architectural Takeaways

1. **Chunk Size Trade-Off:** 300–500 tokens ensures that technical configurations, CLI commands, and explanations remain coherent without diluting the semantic vector embedding.
2. **Vector Space Alignment:** `bge-small-en-v1.5` achieves strong semantic discrimination, separating in-domain technical documentation from out-of-domain noise.
3. **Dense + Sparse Synergy:** While dense embeddings excel at semantic paraphrasing, exact CLI flags (e.g. `--network`, `com.docker.network.bridge.name`) benefit from BM25 sparse keyword scoring, which is why our pipeline combines both via **Reciprocal Rank Fusion (RRF)**.